# v3.10 world spec — the preparation world

*Agreed 5 September 2026. D1 accepted with the rig check reframed; the type-blind analysis folded
in; D2–D6 accepted as recommended. **Implemented**, pre-checks run. Findings from those pre-checks
are at the end, including two that need your call.*

**Why this world.** v3.9 closed the recipe world: a chain whose payoff arrives only at the end is
sparse, and a rare opportunity cannot generate the selection differential that would build the
approach behaviour making it less rare. The fix is not a bigger payoff or a shorter journey — both
were tried — but **a dense opportunity**. Here the fact to be learned sits on **every meal**, where
the agent already is. There is no approach behaviour to evolve, no carrying, no navigation: only
*which action to take on the food under my feet*.

---

## Cells and co-occupancy

Two contents only: **food A** and **food B**, one array pair, at most one type per cell (enforced at
spawn), spawning in drifting patches exactly as v3.1. **No items, no stations, no nuts.** Agents do
not block each other; occupancy is an observation channel only.

## Actions — eight, of which five are live in phase 1

`0–3` move N/S/E/W · `4` **eat** · `5,6,7` **prep_1 / prep_2 / prep_3**

In **phase 1** the three preparation logits are forced to −∞, so phase 1 is **exactly v3.1's five
actions** and its gate applies unchanged. **Only the action space changes at the switch** — no new
input channels, so the transition is one thing, not two.

| action | food A here | food B here | otherwise |
|---|---|---|---|
| **eat** | eat raw: safe → **+`food_value`**, **m = +1**; poison → **−`poison_value`**, **m = −1** | same | no-op, `move_cost` |
| **prep_k** | correct prep for A → **+`prep_value`**, **m = +1**; wrong → **−`prep_fail`**, **m = −1** | correct prep for B → **+`prep_value`**, **m = +1**; wrong → **−`prep_fail`**, **m = −1** | no-op, `move_cost` |

A preparation **consumes the food cell**, exactly as eating does.

**RESOLVED — D1 accepted, and the rig check is reframed.** The consequence is stronger than "noisy":
once the mapping is known, preparation pays 1.5 on *any* food, so the safe/poison fact becomes
**irrelevant** and a good learner stops eating raw entirely. Food learning is then *unselected*, and
`probe_adv` (food) decays for a **good reason** — not because the transition broke `H`.

So **rig check 2(a) is read in the first mapping era of phase 2 only** (steps 0–2000 after the
switch), when nobody knows the mapping and `eat` is still the right action. After that, **prep share
of meals by era** is the diagnostic: a learner that has the mapping should shift from `eat` to
`prep`, and that shift is itself an efficiency signature.

## Modulator events — the complete list

| event | m | energy |
|---|---|---|
| eat safe food | **+1** | +`food_value` |
| eat poison | **−1** | −`poison_value` |
| **correct preparation** | **+1** | +`prep_value` |
| **wrong preparation** | **−1** | −`prep_fail` |
| everything else | **0** | move / no-op `move_cost`, base metabolism, repair |

There is no silent event in this world: every action on food resolves immediately and two-sidedly.
The modulator is **not** the agent's own energy change in general; it is this table.

## Three levels on prep hit — and the type-blind floor

| level | value | what it means |
|---|---|---|
| chance | **1/3** | a random preparation |
| **type-blind** | **0.5** | "always `prep_k`" for a k useful in this era: right for one food type, wrong for the other, **no type knowledge at all**, EV +0.5/meal |
| full | **1.0** | the conjunction: the right preparation for each type |

With distinct mappings, exactly one preparation is useless in any era and each of the other two is
correct for one type. A type-blind policy therefore scores **0.5 within a favourable era** — but
**only 1/3 averaged across eras**, because k is useless in a third of mappings (EV −0.5/meal there).
**A genome beats chance only by tracking the era, not by holding one preparation.**

**A conjunction shows as BOTH types above 0.5**, not one at 1.0 and the other at 0. Per-type hit is
printed for every arm.

## The mapping

Each food type has exactly one correct preparation, so the mapping is a function
{A, B} → {prep_1, prep_2, prep_3}. Drawn at the switch and **redrawn every `prep_every = 2000`
steps**, independently of the safe/poison flip. Chance prep hit = **1/3**.

> **DECISION 2 — must the two food types map to *different* preparations?** If drawn independently,
> they collide one time in three, and a colliding mapping is solvable by learning a single
> preparation and never discriminating food type at all — which is not the task.
> **Recommendation: draw them distinct**, and redraw so that the new mapping differs from the old in
> at least one type (as `new_recipe` already does for the recipe). Chance stays 1/3 per prepared meal.

## Observation — unchanged, and unchanged across the switch

The v3.8/v3.9 layout, 60 inputs. Live: food A / food B / occupancy directional sums and here-values,
plus energy. Nothing tells the agent the mapping; nothing changes at the switch.

> **DECISION 3 — keep the 60-input layout with the chain channels permanently dead, or shrink to the
> ~21 live inputs?** **Recommendation: keep 60.** It costs nothing (dead inputs contribute exactly
> zero), it keeps the phase-1 gate directly comparable with v3.8's and v3.9's, and shrinking would
> change the network size at the same moment as everything else. The dead channels are named in the
> setup print so nobody mistakes them for live.

Hidden 24. No instinct wired: `scaffold_food=False`, `scaffold_chain=False`; `nav_dir` / `nav_here`
stay in the genome, reach nothing, and give the dead-gene drift scale.

## Densities

Food exactly as v3.1 (`spawn_per_patch` 3.0, 8 patches, radius 6, `food_rot` 0.005) — measured
standing cover ~43%. **There is no density to tune in this world**: the opportunity is every food
cell, so the readability criterion that failed three times in v3.9 is satisfied by construction.
Printed in setup for the record, not asserted as a band.

## Phases — one population, never rebuilt

**Phase 1:** 8000 steps, v3.1 unchanged, preparations masked. **Phase 2:** 8000 steps, preparations
live for the same population; agents, `H`, eligibility traces and the world all carry across
untouched. Mapping redrawn at the switch and every 2000 steps thereafter, so phase 2 holds four
whole mapping eras.

## Costs and values

v3.1 metabolism: `base_cost` 0.006 · `move_cost` 0.002 · `start_energy` 1.5 · `founder_energy` 3.0 ·
`repro_threshold` 3.0 · `repro_cost` 1.5 · `max_energy` 5.0 · **`max_pop` 800** · `init_pop` 300 ·
`min_pop` 40 · food +0.7 / poison −0.5 · `flip_every` 300 · `eta_init` 0.2.

Preparation: **`prep_value` 1.0** · **`prep_fail` 0.5** · **`prep_every` 350** · K = 3.
*(v3.11: both changed after the v3.10 acceptance run. See "Changes after the v3.10 acceptance".)*

Expected value of a preparation at chance: `1/3 × 1.0 − 2/3 × 0.5 = **0.000**`, against `+0.1` for a
raw meal at a chance safe rate and `+0.7` for a known-safe one. Knowing the mapping is worth
**+1.5 per meal against +0.7** — and unlike v3.9, the opportunity arrives on **every meal**, so the
differential compounds over a lifetime instead of appearing half a time.

## Arms — five, three seeds

`random policy` · `fixed` · `scrambled` · `plastic (W2)` · `fixed + B (ceiling)`

`random policy` is uniform over the **available** actions — 5 in phase 1, 8 in phase 2 — so the
conditional null is **1/5** then **1/8** per action, and **3/8** for "any preparation". Exempt from
row 0: a random walker belongs at the population floor.

> **DECISION 4 — how does the `B` ceiling act?** In the recipe world B steered navigation and vetoed
> attempts. There is no navigation here; the only choice is which action to take on the food
> underfoot. **Recommendation: B is a hand-wired table over (food type, prep) updated with exact
> credit (`B[f,k] ← 0.7·B[f,k] + 0.3·outcome`), and when the agent stands on food and any entry for
> that type exceeds a threshold, the action is forced to `argmax_k B[f,k]`; otherwise the network
> chooses.** That is the honest analogue of v2's veto: a hand-wired policy override, not a hint. It
> remains a reference level, not a matched comparison, and the summary says so.

## Metrics

**prep hit rate**, event-weighted (correct preps / preps; chance **1/3**) · **P(prep | on food)**
against the null, and per-prep P(prep_k | on food) · **`probe_adv` (prep)**, within-agent, built
like the recipe probe: on a synthetic "food type f here" observation, the logit of the correct prep
minus the mean of the other two, with learned synapses minus innate · **hit-in-life curve** (prep hit
by prep number in an agent's life) and **since-remap curve** (by preps since the last remap) ·
`safe_rate`, raw-meal count and **`probe_adv` (food)** through phase 2 as the rig check · **prepared
meals per life** · `eta2` / `lam2` / `eta1` against `fixed` · unwired `nav_dir` / `nav_here` as drift
scale · transition table at 500-step bins across the switch · populations and injections per phase.

## Stopping rule

Event-weighted, per phase. Margin **0.03**, seeds **3/3**. A positive on any row gets seeds 3–4
before it is called.

| # | row | reading | attributing arm |
|---|---|---|---|
| **0** | uninterpretable per phase: pop < 80, **or injections sustained (mean ≥ 1 per window over the half)**. Plus a **cap check**: no outcome arm above **90% of `max_pop`** in phase 2's second half | exclude and name it. `random policy` exempt. **Why the change:** the v3.10 acceptance excluded phase-2 `fixed` at pop 675/789 against a cap of 800 on 0.1 injections per window in one seed — a near-cap arm is not uninterpretable, and one event in ten windows is not a propped-up population. **If the cap check fails**, raise `max_pop` to 1200 and re-estimate the runtime before reading anything population-dependent | — |
| **1a** | **phase-1 gate = v3.1** — safe rate `plastic` − `fixed` ≥ 0.03, `probe_adv` (food) ≥ 1.0 | **STOP ROW.** If phase 1 is not v3.1, nothing below is read. Row-0 fallback: where `fixed` phase 1 is excluded, that seed reads against v3.1's published range, conservative end 0.56 | `fixed`, or v3.1's published range |
| **1b** | **mapping gate** — `fixed` prep hit **≤ 0.55** in 3/3, **and** first-preparation hit **≤ ~0.55** in `fixed` and `scrambled`. Read the per-type split **PER ERA**, never on the phase half | genes **may** hold a type-blind preparation (0.5); they must not track the **conjunction**. `fixed`'s per-type hit is printed: one type high and the other near 0 is type-blind and allowed; **both above 0.5 in `fixed`** would be genes holding the conjunction. If it fires, shorten `prep_every` toward the flip period, **judged against `fixed` only** | `fixed`, per-type |
| **2** | **rig checks** — (a) food learning survives the switch, **read on safe rate in the first mapping era** (`plastic` − `fixed` ≥ 0.03); the corrected `probe_adv` (food) in the **first 500 steps** after the switch is **corroborating only**, and the **eat → prep shift timing** (prep share in 250-step bins) plus **prep share by era** are printed as diagnostics; (b) the opportunity exists: **prepared meals per life ≥ 3 in `fixed`**; (c) P(prep \| on food) against the per-phase null | if (a) or (b) fails the rig is broken: stop, diagnose, no learner claim. **Why the reframe:** the v3.1 probe form subtracts the best *other* action, and with three preparations live that term moves with food type, so it measured preparation preference, not food learning. The probe is corrected (`eat` minus the mean move logit), but a *decaying* probe after the first era is expected behaviour — once the mapping is known, preparation pays 1.5 on any food and the safe/poison fact stops mattering | `fixed`, `random policy` |
| **3** | **the conjunction.** `plastic` − `fixed` ≥ 0.03 **and** `plastic` − `scrambled` ≥ 0.03 in 3/3 on prep hit; `probe_adv` (prep) > 0; **one within-life signature**; **no abstention** (prepared meals not below 0.8× `fixed`) | **a positive means clearing the type-blind floor.** Read the per-type split: the conjunction is **both types above 0.5**. **`plastic` − `scrambled` on hit rate is a survivorship-contaminated contrast** — agents whose random `H` happens to help live longer, enriching the standing population with nothing learned. It stays required, but the **attribution** is carried by the within-agent lines: `probe_adv` (prep) and row 3a | `scrambled` carries the claim; `probe_adv` is within-agent; abstention read first |
| **3a** | **survivorship diagnostics (required).** (i) **survivor curve** — hit on preparations 1–5 vs 6–10 over agents that reached 10 preparations: **rising in `plastic`, flat in `scrambled`**. Every agent counted contributes both halves of its own curve, so a rise is the same individuals later in their own lives, not a different sample. (ii) **first-preparation hit** — the agent has learned nothing, so this reads the innate policy the standing population carries | a `plastic` − `scrambled` gap on hit rate with a **flat** survivor curve is enrichment, not learning, and row 3 is **not** called positive on it. Note `probe_adv` is computed over the **living** and so is itself partly survivorship-selected; the survivor curve is not | within-agent |
| **3b** | **knockout** — late `plastic` and `scrambled` genomes replayed with `eta_scale = 0` in a **fresh world seed**: same brains, no learning. **Read the FIRST 500 steps, not the second half.** `eta_scale = 0` stops learning but **not reproduction**: measured on the v3.10 acceptance, `max_gen` went 4–8 → 9–35 over 3000 steps and the hit rate climbed with it (`plastic` seed 0: 0.485 → 0.657), so the second half measures **re-selection in the new world**, not the genome. Both windows and both `max_gen` values print, so the contamination stays visible | if the standing advantage lives in `H`, **both** fall to the type-blind floor. What then separates them is the survivor curve, which is learning rather than luck. A `plastic` genome that keeps its advantage with learning off had it in the **genome**, not in `H` | within-genome |
| **4** | gene rows: `eta2`, `lam2`, `eta1` against `fixed` and `scrambled`; unwired scaffold genes as drift scale | corroborating only | — |
| **5** | if row 3 is null with rows 1–2 clean | the first earned statement about the learner's limit, on a dense, immediate, two-sided task. Spend the one rule-form change there | — |

**Pre-registered prediction, refined after the v3.10 acceptance and recorded before the v3.11
run:**

1. **`fixed` at 0.52–0.56** — its own type-blind level and no more. At `prep_every` = 700 gate 1b
   still fired (first-prep hit 0.67–0.72), because of **standing polymorphism**: the population
   carries genotypes for several of the six possible mappings at once, so a remap needs no
   mutation — lineage selection just promotes whichever genotype already matches. 350 is below a
   generation, so a lineage cannot be selected up within an era.
2. **`scrambled` ≈ `fixed`.** Its elevation in v3.10 was the same genetic era-tracking, not lucky
   `H`: survivor halves matched `fixed` to three decimals and first-prep hit was 0.821/0.607. It
   should track `fixed` again, and its `probe_adv` should stay at zero.
3. **`plastic (W2)` at 0.65–0.70**, and above 0.5 on **both types** — the conjunction, not a
   type-blind guess. It should lose less to the shorter era than `fixed` does, because it
   relearns within a life rather than waiting on a generation.
4. **The since-remap curve dips and recovers within ~5 preparations** in `plastic`: at a remap the
   learned `H` is wrong, so a learner pays for the change and earns it back. Flat means nothing is
   relearned within a life; never recovering means 700 steps is shorter than the learner needs,
   which would itself be the answer to whether any era length separates selection from learning.

> **DECISION 5 — the within-life signature is harder here than it looks.** A mapping era is 2000
> steps and a generation is ~200, so most agents live inside a single era and never see a remap.
> `hit_old` > `hit_young` and the hit-in-life curve therefore measure learning *within* an era, which
> is what we want — but an agent born mid-era inherits nothing and must learn from its own
> preparations. **Recommendation: keep `prep_every` at 2000 and read the since-remap curve
> population-wide**, which is where a remap's cost and recovery show. If gate 1b fires and
> `prep_every` shortens, both curves become more informative, not less.

> **DECISION 6 — the semantics test needs the mapping in the loop.** The v3.9 test enumerated
> (action, cell content, inventory). Here the third dimension is the **mapping**.
> **Recommendation: enumerate every (action ∈ {eat, prep_1..3}, food ∈ {none, A, B}, mapping ∈ all
> distinct assignments) row** — 4 × 3 × 6 = 72 rows plus the four move/no-op rows — constructing the
> cell, calling one `resolve_action`, and asserting energy delta, `m`, event and that the food cell
> was consumed. It runs well under a second and it is the test that would catch a mapping applied to
> the wrong food type.

## Also carried over unchanged

The learning rule and its genes; the staging mechanism; event-weighted aggregation per phase; the
conditional-null convention (measured null printed beside the analytic one); the learning-rule unit
test in the setup cell; QUICK mode; Colab-only with `sim.py` and `analysis.py` uploaded alongside;
per-run pickling with resume instructions.


---

# Findings from the pre-checks (1 seed, 3000-step phases, every arm)

**The world is readable — the thing v3.9 never achieved.** Prepared meals per life: `fixed` **16.5**,
`plastic` **19.7**, ceiling **62.2**, null **6.6**, against a criterion of ≥ 3. There is no density
to tune and no approach behaviour to evolve.

| arm | P2 pop | prep hit | hit \| A | hit \| B | prep/life | probe_adv (prep) |
|---|---|---|---|---|---|---|
| random policy | 117 | **0.334** | 0.337 | 0.330 | 6.6 | — |
| fixed | 338 | **0.500** | 0.740 | 0.256 | 16.5 | — |
| scrambled | 399 | 0.722 | 0.553 | 0.840 | 15.8 | **0.342** |
| plastic (W2) | 399 | 0.833 | 0.934 | 0.692 | 19.7 | **5.624** |
| fixed + B (ceiling) | 399 | 0.886 | 0.879 | 0.894 | 62.2 | — |

`random policy` lands on chance and **`fixed` lands on exactly the type-blind floor, 0.500, with
A 0.740 / B 0.256** — the predicted signature, and gate 1b is clear at ≤ 0.55.

### Three things that need your call

**1. `scrambled` sits well above the type-blind floor (0.722, both types above 0.5).** The control is
showing what looks like conjunction knowledge. Its **within-agent probe is 0.342 against plastic's
5.624**, a 16× gap, so the population-level hit rate carries a **survivorship** component the probe
does not: agents whose random `H` happens to favour the correct preparation live longer, so the
standing population is enriched for lucky `H` without anything being learned. Row 3 already requires
both lines, which is the right design — but the hit-rate margin against `scrambled` is partly a
survivorship comparison and should be read that way.

**2. Rig check 2(a) fails as specified, and I found the instrument at fault first.** The v3.1 food
probe is `logit(eat)` minus the **best other action** — and with three preparations live, that term
moves with food type, so the probe was measuring preparation preference. Fixed: the probe is now
`logit(eat)` against the **move** logits, which are food-type-independent. That raised the first-era
value from 0.040 to **0.196** — still far below the ≥ 1.0 criterion. The cause is that the eat→prep
shift happens **inside** the first era (prep share 0.836 in era 1), so the window the criterion
assumes is shorter than 2000 steps. Safe rate does hold there: `plastic` **0.599** against `fixed`
0.489. **Proposal:** read 2(a) on **safe rate in the first era** (`plastic` − `fixed` ≥ 0.03) with the
probe corroborating, or read the probe in the first ~500 steps after the switch. I have not measured
the 500-step version.

**3. Populations sit at the 400 cap** in three of five arms. At a hard cap births become a queue
rather than differential fecundity, which blunts selection — the same issue that cost a v3.6 tuning
pass. Raising `max_pop` is the fix; it is a spec change and yours to make.


---

## Changes after the v3.10 acceptance run

Two parameter changes, each with its own check, and three rule fixes. Nothing here was judged
against a plastic condition's outcome.

| change | from → to | why | check |
|---|---|---|---|
| `prep_every` | 2000 → **700** | at 2000 there are ~12 generations per era and selection uses them: `fixed`'s `prep_gain innate` was **0.995 / 1.611** against `random policy`'s −0.10, and its per-era split had both types above 0.5 in 2 of 4 eras | `fixed` prep hit **≤ 0.55**, and `fixed` / `scrambled` **first-prep hit ≤ ~0.55** |
| `prep_value` | 1.5 → **1.0** | at 1.5 a *chance* preparation paid **+0.167**, so a population could ride the preparation payoff without knowing anything, and every outcome arm sat at 675–799 against a cap of 800. At 1.0 the chance EV is exactly **0.000** — value comes only through knowledge of the mapping | no outcome arm above **90% of `max_pop`** in phase 2's second half; if it still caps, `max_pop` → 1200 and re-estimate the runtime |

The economy after the change: a chance preparation is worth **0.00**, eating raw at chance
**+0.10**, knowing the flip **+0.70**, knowing the mapping **+1.00**. Preparation is now strictly
worse than raw eating until the mapping is known, and strictly better once it is.

Rule fixes: **row 0** as above · **row 3b** reads its first 500 steps · **the pre-registered
prediction** refined to four numbered clauses.


---

## Changes after the v3.11 acceptance run at `prep_every` 700

Every within-life line was positive in 2/2, and gate 1b still fired.

**Why it fired: standing polymorphism.** There are six possible distinct mappings, and the
population carries genotypes for several of them at once. A remap therefore needs no mutation and
no new adaptation — **lineage selection simply promotes whichever genotype already matches**, and
it can do that inside a single era. The signature is in the per-era `(A, B)` pairs, which *flip*
between eras rather than drifting, and in a first-preparation hit of **0.67–0.72**: that is the
innate policy of the standing population, measured before the agent has learned anything.

| change | from → to | why | check |
|---|---|---|---|
| `prep_every` | 700 → **350** | below a generation, so a matching lineage cannot be selected up within an era. Deliberately **not a multiple of `flip_every` = 300**, so the fast fact and the slow fact do not come into phase | gate 1b re-read at 2 seeds. **If it still fires, the next change is K = 4 preparations** — which takes the number of distinct mappings from 6 to 12 and makes standing polymorphism across all of them much more expensive |

Rule changes, all in the reading and none in the world:

- **Rig check 2(a)** reads on **whole-phase founder-free safe rate**, `plastic` − `fixed` ≥ 0.03.
  At 350 an era is far too few meal events to read a rate on.
- **Abstention** fires only if prepared meals per life < 0.8× `fixed` **and** (hit ≤ `fixed`
  **or** pop ≤ `fixed`). Preparing less while scoring and living better is a learner declining bad
  bets, not one abstaining from the task.
- **Survivor curve** halves become preparations **1–2 against 6–10**: the first two are before
  within-life learning could have taken hold, so it is the agent's own naive rate against its own
  settled rate.
- **New: survivor-conditioned since-remap curve.** Only agents that made **8 preparations both
  before and after the same remap** are counted, each contributing its own rate either side. The
  population-level since-remap curve is open to the objection that the agents alive at preparation
  1 are not the ones alive at 10; this line is not.


## 1. Setup

The sim and the analysis are imported, not inlined. This cell fails loudly if either file is missing.

In [ ]:
# --- Colab check -------------------------------------------------------------
# Upload sim.py and analysis.py next to this notebook (Files pane, or run:
#     from google.colab import files; files.upload()
# and pick both).  Nothing else is needed: pure numpy + matplotlib.
import os, sys

missing = [f for f in ("sim.py", "analysis.py") if not os.path.exists(f)]
if missing:
    raise SystemExit(f"missing {missing} in {os.getcwd()} -- upload them next to this notebook")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import sim, analysis as A

print("numpy", np.__version__)
print("observation size:", sim.N_IN, " actions:", sim.N_ACTIONS,
      " (eat =", sim.EAT, ", preps =", sim.PREP0, "..", sim.PREP0 + sim.N_PREPS - 1, ")",
      " hidden:", A.WORLD["hidden"], " chance recipe hit:", round(A.CHANCE, 3))
print("\nworld (v3.1 metabolism throughout; the chain from v3.6):")
for k, v in A.WORLD.items():
    print(f"  {k:<20} {v}")
print("\n--- world-semantics self-test (every row of the action x cell table) ---")
if not sim.world_semantics_selftest():
    raise SystemExit("the world does not match the spec; nothing below is meaningful")

print("\n--- density ---")
print("no density to assert: the opportunity is every food cell")

print("\n--- founder-tag self-test ---")
print("injected agents are fresh random genomes; their OWN events are excluded from every")
print("event-weighted metric, their children's are not.  The test forces injection so the")
print("exclusion path is actually exercised -- passing on a run with no injections proves nothing.")
if not sim.founder_tag_selftest():
    raise SystemExit("the founder tag is not wired correctly; every founder-free number is suspect")

print("\n--- replay-mapping self-test ---")
print("a knockout that re-seeds the world does NOT get the mapping its genomes were selected")
print("under.  This checks that force_mapping pins it, and that a re-seeded replay can differ")
print("from the source's final mapping -- the condition that made the old knockout misread.")
if not sim.replay_mapping_selftest():
    raise SystemExit("the replay does not carry the mapping it is given; row 3b is meaningless")

print("\n--- frozen-replay self-test ---")
print("row 3b replays an era-boundary snapshot with births, deaths and injection disabled, so")
print("nothing can change but H.  With learning OFF the hit rate must not move across the")
print("window; if it does, the eta 1 side cannot be read as learning.")
if not A.frozen_selftest():
    raise SystemExit("the frozen replay is not frozen; row 3b would not be an attribution")

print("\n--- learning-rule self-test ---")
print("one agent, one fixed observation, one chosen action; action_noise = 0 so act() is")
print("deterministic and the logit checked belongs to the action that laid the trace.")
if not sim.learning_rule_selftest():
    raise SystemExit("the learning rule is not behaving; nothing below is meaningful")

print("\nconditions:")
for name, spec in A.VARIANTS.items():
    ph = " -> ".join(f"{p['n_steps']} steps chain={p['chain']}" for p in spec["phases"])
    print(f"  {name:<24} {ph}")
    print(f"  {'':<24} {({k: v for k, v in spec['kw'].items() if k not in A.WORLD})}")


## 2. Run — staged

`MODE` picks the stage. All stages share **one checkpoint** and each **skips any (arm, seed) already in it**, so `"grid"` continues from the acceptance checkpoint rather than redoing it.

| stage | arms | seeds | runs | est. wall clock |
|---|---|---|---|---|
| `quick` | 3 core | 0 | 3 | ~5 min |
| `acceptance` | fixed, scrambled, plastic (W2) | 0–1 | 6 | ~60–75 min |
| `grid` | + random policy, ceiling | 0–2 | 9 more | **~75–90 min** |

### Read the checkpoint audit before you read anything else

The cell prints an audit of the checkpoint it loads. Runs written **before** a field existed cannot be re-analysed for the reads that depend on it — those counters are accumulated inside the sim, not derived from the log — and `final_mapping` is not reconstructable offline at all, because `World` shares its rng with the agents, so the mapping draw sequence depends on every action-noise draw in the run.

The affected reads are **row 3b (the attribution line)**, **first-preparation hit late-in-era**, **the survivor-conditioned since-remap curve**, and **the survivor halves 1–2 vs 6–10** (older runs recorded the 1–5 split under the same name). They print `nan` rather than a wrong number.

To recover them, put the affected pairs in `REFRESH` — for the v3.11 acceptance checkpoint that is the three core arms at seeds 0–1, six runs, roughly an extra 60–75 min. Everything else (founder-free hit rates, rig check 2(a) on whole-phase safe rate, the abstention rule, the per-era A+B sums, `prep_gain innate`) reads correctly off the existing checkpoint without a refresh.

The cell checkpoints after **every run**, so a dropped session costs one run; just re-run it to resume. **Seeds 3–4 are held in reserve** — set `SEEDS = [3, 4]` after the grid and re-run; the seed criterion adapts (`min(4, n_seeds)`).

In [ ]:
# MODE picks the stage.  All stages share ONE checkpoint and each SKIPS any (arm, seed)
# already in it, so "grid" continues from the acceptance checkpoint rather than redoing it.
#
#   "quick"       smoke test, 3 arms, 1 seed, 1500-step phases.            ~5 min
#   "acceptance"  fixed / scrambled / plastic (W2) x seeds 0-1, full.      ~60-75 min
#   "grid"        continues: adds `random policy` and `fixed + B (ceiling)`
#                 for seeds 0-1, and all five arms for seed 2.  Nine runs.  ~75-90 min
MODE = "grid"

CKPT = "results_v3_11.pkl"
CORE = ["fixed", "scrambled", "plastic (W2)"]
ALL  = list(A.VARIANTS)

# REFRESH: (arm, seed) pairs to re-run even though the checkpoint has them.  Fill this from the
# audit printed below.  A checkpoint written before a field exists CANNOT be re-analysed for the
# reads that depend on it -- those counters are accumulated inside the sim, not derived from the
# log -- and `final_mapping` in particular is not reconstructable offline, because World shares
# its rng with the agents.  Leave empty to continue without those lines (they print `nan`).
REFRESH = []
# REFRESH = [(a, s) for a in CORE for s in (0, 1)]     # <- to recover every read-side line

if MODE == "quick":
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0], 1500, CORE, dict(prep_every=300, recipe_every=500)
    CKPT = "results_v3_11_quick.pkl"
elif MODE == "acceptance":
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0, 1], 8000, CORE, {}
elif MODE == "grid":
    SEEDS, PHASE_STEPS, ARMS, OVERRIDES = [0, 1, 2], 8000, ALL, {}   # seeds 3-4 held in reserve
else:
    raise SystemExit(f"MODE must be quick / acceptance / grid, not {MODE!r}")

existing = A.load(CKPT)
if existing:
    A.checkpoint_audit(existing)
    print()

variants = {k: A.VARIANTS[k] for k in ARMS}
results = A.run_experiment(seeds=SEEDS, phase_steps=PHASE_STEPS, variants=variants,
                           results=existing, save_path=CKPT, refresh=REFRESH, **OVERRIDES)
print("\nin the checkpoint:", {k: sorted(r["cfg"]["seed"] for r in v) for k, v in results.items()})


## 3. The printed summary

Phase 1 and phase 2 blocks separately, then per-seed values, then the decision numbers in the order of the table above.

**Read row 1 first — it is a stop condition.** If phase 1 does not reproduce v3.1, nothing in phase 2 is readable.

**Reading a QUICK run:** a smoke test, not a result. Phases of 1500 steps are a handful of generations, so row 0 will flag conditions and row 1 will not reproduce. Read it only to confirm every cell produces the output it should.

In [ ]:
A.checkpoint_audit(results)
A.summary(results)


## 4. The transition

500-step bins across 2000 steps either side of the chain switching on. This is where a collapse becomes visible, and where `eta2`/`lam2` either hold or start falling as they did in v3.6.

In [ ]:
A.transition_table(results, bin_size=500, span=2000)


## 5. Curves

`meal` is v3.1's within-life food curve, read on **phase 1** — the positive control. `att` and `rec` are the recipe curves, read on **phase 2**; row 4 requires `att` to rise (or `hit_old` > `hit_young`).

In [ ]:
A.curves(results, "meal", "meal number in an agent's life")        # phase 1
A.curves(results, "att",  "attempt number in an agent's life")     # phase 2
A.curves(results, "rec",  "attempts since the last recipe change")  # phase 2


## 6. Plots

Solid black line = the chain switches on; dashed = recipe change.

In [ ]:
A.plot_results(results)


## 8. Reading it

**Rows 1a and 2 are stop conditions.**

1. **Row 1a** — phase 1 must be v3.1. If not, nothing below is readable.
2. **Row 1b** — `fixed` must not hold the *conjunction*. A type-blind 0.5 is allowed; read the
   per-type split to tell them apart.
3. **Row 2** — food learning in the first mapping era **read on safe rate** (`plastic` − `fixed`
   ≥ 0.03), with the corrected `probe_adv` (food) in the first 500 steps corroborating; then
   prepared meals per life ≥ 3 in `fixed`, and P(prep | on food) against the null. The
   **eat → prep shift timing** and **prep share by era** are printed as diagnostics, not gates.
   A probe that decays *after* the first era is expected: once the mapping is known, preparation
   pays on any food and the safe/poison fact stops mattering.
4. **Row 3** — the abstention line first, then the two result lines, then **the per-type split**:
   a conjunction is *both* types above 0.5, not one at 1.0 and one at 0.
5. **Rows 3a and 3b — survivorship.** `plastic` − `scrambled` on hit rate is a *contaminated*
   contrast: agents whose random `H` happens to help live longer, enriching the standing
   population with nothing learned. It stays required, but the attribution is carried by the
   within-agent lines. The **survivor curve** (preps 1–5 vs 6–10 over agents that reached 10)
   must **rise in `plastic` and stay flat in `scrambled`** — every agent counted contributes both
   halves of its own curve, so a rise is the same individuals later in their own lives. The
   **knockout** replays late genomes with `eta_scale = 0` in a fresh world: if the advantage lives
   in `H`, both arms fall to the type-blind floor. Note `probe_adv` is computed over the *living*
   and so is itself partly survivorship-selected; the survivor curve is not. The pre-checks had
   `scrambled` at 0.722 on hit rate but only 0.342 on the probe, against plastic's 5.624 — that
   gap is what these rows are here to adjudicate.
6. **Row 5** — a null here, with rows 1–2 clean, is the first earned statement about the learner's
   limit: no approach behaviour required, credit immediate and two-sided, opportunity on every meal.

---

## 9. What the v3.10 acceptance run established (2 seeds, full phases)

These two are settled, and both are recorded here because they change what the controls mean.

### Row 1b fired, and the cause is genetic era-tracking — not luck, not a broken floor

`fixed` prep hit was 0.625 and 0.699 against a gate of 0.55. Type encounters are balanced
(share of preparations on type A: 0.499 / 0.498 in `random policy`, 0.435–0.541 elsewhere), so
the 0.5 type-blind floor constant is right and the excess is not a rig asymmetry. The elevation
is arithmetic on the per-type split:

    seed 0:  0.625 = 0.5 x 0.872 + 0.5 x 0.377

and the second term is the finding. A pure type-blind genome — "always prep_k" — scores **0** on
the wrong type. 0.377 is not 0. So the genome is not type-blind: it holds a **partial genetic
conjunction**, and the phase-half aggregate hid it because it spans two mapping eras with
different mappings, averaging a *switch* into a false one-high-one-low reading. Per era:

    fixed seed 0  (A,B)  [(0.72, 0.93), (0.56, 0.96), (0.95, 0.34), (0.77, 0.43)]
                  both types above 0.5 in 2 of 4 eras

`prep_gain innate` — the genome's preference for the correct preparation on a synthetic
observation, with no gating in it — confirms it directly: `fixed` **0.995, 1.611**, against
`random policy`'s **−0.102, −0.128**. At `prep_every` = 2000 there are ~12 generations per era
and selection uses them. Hence the change to 700.

### `scrambled`'s elevation is the same mechanism, not lucky H

`scrambled` reached 0.746 / 0.575 on prep hit, which invites reading it as survivorship — agents
whose random `H` happens to help living longer. It is not. Three lines say so:

- **Survivor halves match `fixed` to three decimals.** `scrambled` 0.815 → 0.724, `fixed`
  0.812 → 0.721. An arm whose advantage came from lucky `H` would not track the arm that has no
  `H` at all.
- **First-preparation hit is 0.821 / 0.607** — high, and that measurement is taken before the
  agent has learned anything, so it reads the genome and nothing else.
- **`prep_gain` innate is 0.859 / 0.446**, close to `fixed`'s and far above `random policy`'s
  zero, while its learned `probe_adv` is **−0.008 / 0.114** — essentially nil.

So `scrambled` is a `fixed`-like genome carrying a scrambled, useless `H`. That is exactly what
the control is supposed to be, and it means `plastic − scrambled` is a cleaner contrast than the
survivorship worry suggested — but it also means `scrambled` inherits `fixed`'s era-tracking, so
it is subject to the same gate 1b.

### The route that separates the arms

`plastic` and `fixed` reached comparable hit rates by **opposite routes**, and this is what the
whole rig is now built to read:

| | plastic | fixed |
|---|---|---|
| first-preparation hit (genome) | 0.458, 0.256 | 0.648, 0.761 |
| `prep_gain` innate (genome) | 0.324, −0.149 | 0.995, 1.611 |
| knockout, first window (genome) | 0.485, 0.646 | — |
| `probe_adv` (prep), learned | 6.52, 6.84 | n/a |
| live phase-2 hit | 0.810, 0.715 | 0.625, 0.699 |

`plastic` learns it within life on a genome that is **worse** than `fixed`'s; `fixed` evolves it
into the genome. Four instruments, two of them within-agent, all agreeing.

---

## 10. Founder dilution, and why a floor population is not a broken arm

When a population falls to `min_pop` the world injects fresh **random** genomes to hold it off the
floor. Those agents forage and prepare like anyone else, and their events land in the same
event-weighted totals as everyone's — so an arm that needs injecting has its metrics pulled toward
chance **in proportion to how badly it is doing**. A non-learner therefore reads as *more random*
the worse it does, which is a metric artifact, not a fact about the arm. That is founder dilution,
and it is what made phase-2 `fixed` look uninterpretable in the v3.11 pre-check.

The fix is at the metric, not the world. Injected agents carry `injected=True`; **their children do
not** — a founder's descendants are ordinary selected lineages and count from the first generation.
Every event-weighted number is reported **founder-free**, with the injected agents' own events
excluded: prep hit, per-type hit, survivor curve, since-remap curve, first-preparation hit and safe
rate. The **founder share of events** prints per arm and per phase so the size of the removed
dilution is visible, and the all-agents version prints alongside for this build.

**Row 0 excludes on `pop < 80` over the half and nothing else. Injections are reported, not
exclusionary.**

### The verdict this makes readable

A non-learning population sits at the floor in this world **because value comes only through
knowledge.** With `prep_value` 1.0 against `prep_fail` 0.5 the chance EV of a preparation is exactly
zero, and raw eating pays +0.10 at chance against +0.70 knowing the flip. An arm that learns
neither fact has no income to grow on. That is not a rig failure — it is **the same verdict v3.1
gave**, arrived at again in a world where the fact to be learned sits on every meal. The v3.10
`prep_value` of 1.5 had been concealing it by paying a *chance* preparation +0.167, which let
`fixed` grow to the population cap on knowledge it did not have.

The **row-0 fallback for phase 1 stays as is**: where phase-1 `fixed` is excluded on population,
that seed's row 1a reads against v3.1's published range, conservative end 0.56.

### One consequence for reading the gates

Once encounters are skewed, `max(share_A, share_B)` — not 0.5 — is the level a type-blind policy
reaches, because "always prep_k" for the commoner type beats 0.5 with no type knowledge at all.
Each arm's own type-blind level now prints beside its hit rate, and gate 1b is read against it.

---

## 11. Why gate 1b fired at `prep_every` = 700: standing polymorphism

The v3.11 acceptance had **every within-life line positive in 2/2** and gate 1b firing anyway. The
mechanism is not mutation and not within-life learning in `fixed`. It is **standing polymorphism**.

There are only **six** distinct mappings of two food types onto three preparations. A population of
several hundred carries genotypes for several of them **at the same time**. So when the mapping is
redrawn, nothing has to be invented: **lineage selection promotes whichever genotype already
matches**, and at 700 steps an era is long enough — more than a generation — for it to do so.

Two signatures, both in the printed output:

- **The per-era `(A, B)` pairs flip between eras** rather than drifting. A genome slowly acquiring
  a conjunction would improve monotonically; a population switching between standing genotypes
  shows the high type jumping from A to B and back as the mapping moves.
- **First-preparation hit of 0.67–0.72.** That is measured on an agent's very first preparation,
  before it has learned anything, so it reads the innate policy the standing population carries —
  and it is well above the type-blind level.

This is why `prep_every` goes to **350**: below a generation, so a matching lineage cannot be
selected up inside an era. It is also deliberately **not a multiple of `flip_every` = 300**, so the
fast fact (which type is safe) and the slow fact (which preparation goes with which type) do not
come into phase and cannot be tracked by a single periodic cue.

**If the gate still fires at 350, the next change is K = 4 preparations.** That takes the number of
distinct mappings from 6 to 12, which makes carrying standing genotypes for all of them
substantially more expensive — it attacks the mechanism directly rather than the time available to
it.

---

## 12. The v3.11 acceptance at `prep_every` 700, and why 350 was reverted

**Every within-life line was positive in 2/2, and gate 1b fired anyway.** The mechanism is now
measured rather than inferred, and it is not one a shorter era can beat.

### Genes track the mapping by survival sorting over standing variation

There are only **six** distinct mappings. A population of several hundred carries genotypes for
several of them at once, so a remap requires no mutation and no adaptation: **survival sorting
promotes whichever genotype already matches**, and it completes well inside a third of an era.

| measurement | value |
|---|---|
| `fixed` first-preparation hit, **late in the era** | **0.82** |
| genome-only hit (`eta = 0`), **matched** mapping | **0.844** (A 0.899, B 0.798) |
| genome-only hit (`eta = 0`), **shuffled** mapping | **0.296** (A 0.546, B 0.093) |

The pair is the whole story: the genome holds the conjunction **for the mapping it was sorted
under, and only for that one** — below chance on the swap.

### 350 shortened the learner's payoff window without touching the sorting

The pre-registered prediction failed in **both** directions:

| | predicted at 350 | observed |
|---|---|---|
| `fixed` | 0.52–0.56 | **0.642** |
| `plastic (W2)` | 0.65–0.70 | **0.575** |

So the learner lost to the non-learner, and `prep_gain innate` for `fixed` went *up* over the
change (0.149 at 700 → 0.379 at 350). Sorting is not rate-limited by generations — only by how
fast the mismatched fraction dies, which is fast. **`prep_every` is back to 700**, and the fix is
the size of the mapping space, not the speed of the world: see `spec_v3_12.md`.

### Gate 1b is now a measured genetic baseline, not a stop

The row no longer halts the reading. It reports how much of the standing hit rate the genome
already carries — `fixed`'s late first-preparation hit, the per-era A+B sum, and row 3b's
matched/shuffled genome hit with learning off — and **the learner's contribution is read above
it**.

### Row 3b is the attribution line

For each `plastic` seed, the late genomes are replayed on the **matched** mapping and on a
**shuffled** one, with learning **off** (`eta 0`) and **on** (`eta 1`), in a single 10-step window,
with `pop` and `max_gen` beside every number.

**The shuffled pair carries the claim.** On a mapping no genotype was sorted for, learning-off is
the genetic floor and learning-on is what the rule adds within a life.
**Required: `eta 1` − `eta 0` ≥ 0.10 on the shuffled mapping in every seed.**

The window is 10 steps because 50 was not short enough — `max_gen` reached 4.0 inside it, which is
four generations of selection on the pinned mapping, i.e. sorting rather than the genome.
`knockout_window_selftest` holds the line: a **non-plastic** genome's per-type hits must swap when
the mapping swaps, which they do not if the window allows re-evolution.

---

## 13. The v3.11 grid, read — and row 3b's failure as an instrument

**Rows 1a, 2 and 3 pass 3/3.** The full write-up with per-seed numbers is `v3_11_finding.md`.

### What carries the attribution

Not the population hit rate. Two within-agent lines:

- **Survivor curve** (preparations 1–2 vs 6–10, over agents that reached 10; every agent
  contributes both halves of its own curve): **rising in `plastic (W2)` 3/3, falling in the
  controls 3/3.**
- **First-preparation hit, late in the era** (the genome, before anything is learned):
  **`plastic` sits at its type-blind level; the controls sit at 0.73–0.82.**

The inversion is the result: **the learner's genome is the worst of the arms and its standing
performance is built within life; the non-learners' genomes are the best and theirs is built by
selection.** Comparable places, opposite routes.

### Row 3b failed as an instrument — no conclusion was drawn from it

It did not return a negative. It could not return anything, for two independent reasons, both mine:

1. **A 10-step window cannot show learning that takes ~5 preparations.** The since-remap curve
   recovers by preparation 5; in 10 steps an agent makes one or two. The window had been cut to 10
   to stop the replayed population re-evolving — and in fixing that, I cut it below the timescale
   of the thing being measured.
2. **A run-end snapshot sits mid-era and is only partly sorted.** The run ends ~300 steps into an
   era. `fixed` seed 0 scored **0.24 on its own mapping**, when a genome selected under that
   mapping should be near its type-blind level.

### The cap check, and the population column

The cap check is recorded: no outcome arm above 90% of `max_pop` = 800 in phase 2's second half.

**The population column is not read.** Population is an outcome of the economy, not a measure of
the learner, and across v3.10–v3.11 it was twice the thing that moved when a parameter changed — to
the cap at `prep_value` 1.5, to the floor at 1.0. It is reported, it gates row 0 at `pop < 80`, and
nothing in the claim rests on it.

## 14. Row 3b rebuilt: the frozen-population replay

Genomes are snapshotted at **every era boundary**, so the population has just lived a whole era
under that mapping and is sorted for it. One snapshot is replayed for **300 steps** with **births,
deaths and injection all disabled** — energy is tracked and spent, the metabolism runs, it is
simply not lethal — on the **matched** mapping and on a **shuffled** one, with learning **off**
(`eta 0`) and **on** (`eta 1`).

**Nothing can change over the window except `H`.** Not the population's composition, not its size,
not which lineages are present. A hit rate that moves under `eta 1` and does not move under `eta 0`
is within-life learning and can be nothing else — not sorting, not survivorship, not founder
replacement.

**Requirement: `eta1 − eta0` ≥ 0.10 on the shuffled mapping in every seed.**

`analysis.frozen_selftest` guards it, and it is the test the old knockout never had. Measured on a
`plastic` run at the last era boundary:

```
eta 0: hit 0.688 -> 0.678  (drift +0.009)   pop [300]   max_gen [0]
eta 1: hit 0.773 -> 0.892  (drift +0.118)   pop [300]   max_gen [0]
```

`pop` and `max_gen` are single-valued across the window: no agent was born, none died. With
learning off the hit does not move; with it on, it climbs.

This is **v3.12's D6 claim line**. The v3.11 `plastic` arm will be re-run at 3 seeds with per-remap
snapshots as an **addendum** once the v3.12 build is done. It gates nothing.
